In [7]:
!pip install -qU langchain langchain-community requests==2.32.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.3.84 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.


In [8]:
from langchain.prompts import PromptTemplate

# Defining the template
template_1 = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

# Reusing the template
print(template_1.format(topic="Quantum Physics"))


Explain Quantum Physics in simple terms for beginners


In [9]:
multi_input_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

test_cases = [
    {"topic": "AI", "audience": "beginners", "tone": "friendly"},
    {"topic": "Python", "audience": "kids", "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers", "tone": "technical"}
]

for case in test_cases:
    print(multi_input_template.format(**case))

Explain AI for beginners in a friendly tone
Explain Python for kids in a fun tone
Explain Deep Learning for engineers in a technical tone


In [10]:
variations = {
    "Teaching": PromptTemplate.from_template("Explain {topic} clearly step by step"),
    "Interview": PromptTemplate.from_template("Ask 3 questions about {topic}"),
    "Storytelling": PromptTemplate.from_template("Explain {topic} as a story")
}

topic_input = "Machine Learning"

for style, tmplt in variations.items():
    print(f"--- {style} Variation ---")
    print(tmplt.format(topic=topic_input))

--- Teaching Variation ---
Explain Machine Learning clearly step by step
--- Interview Variation ---
Ask 3 questions about Machine Learning
--- Storytelling Variation ---
Explain Machine Learning as a story


In [11]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

def get_chat_prompt(role_type, topic_name):
    roles = {
        "teacher": "You are a helpful assistant that explains complex concepts simply.",
        "interviewer": "You are a technical interviewer assessing candidate knowledge.",
        "motivator": "You are an inspiring mentor who encourages learners."
    }

    system_message = SystemMessagePromptTemplate.from_template(roles.get(role_type, "You are a helpful assistant."))
    human_message = HumanMessagePromptTemplate.from_template("Tell me about {topic}.")

    chat_template = ChatPromptTemplate.from_messages([system_message, human_message])
    return chat_template.format_prompt(topic=topic_name).to_messages()

# Example usage
print(get_chat_prompt("teacher", "Neural Networks"))

[SystemMessage(content='You are a helpful assistant that explains complex concepts simply.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about Neural Networks.', additional_kwargs={}, response_metadata={})]


In [12]:
def validate_inputs(audience, tone):
    valid_audiences = ["beginner", "intermediate", "expert"]
    valid_tones = ["formal", "casual", "fun"]

    # Validation logic with default fallbacks
    final_audience = audience if audience in valid_audiences else "beginner"
    final_tone = tone if tone in valid_tones else "formal"

    return final_audience, final_tone

# Test validation
a, t = validate_inputs("pro", "silly")
print(f"Validated values: Audience={a}, Tone={t}")

Validated values: Audience=beginner, Tone=formal


In [13]:
def generate_prompt(topic, audience, tone, style):
    # 1. Validate
    v_audience, v_tone = validate_inputs(audience, tone)

    # 2. Select Style Template
    styles = {
        "teaching": "Explain {topic} for {audience} in a {tone} teaching style.",
        "interview": "Generate interview questions about {topic} for an {audience} candidate in a {tone} tone.",
        "storytelling": "Explain {topic} for {audience} in a {tone} storytelling style."
    }

    selected_template = PromptTemplate.from_template(styles.get(style, styles["teaching"]))

    # 3. Generate
    return selected_template.format(topic=topic, audience=v_audience, tone=v_tone)

print(generate_prompt("Neural Networks", "beginner", "fun", "storytelling"))

Explain Neural Networks for beginner in a fun storytelling style.


In [14]:
reusable_tmplt = PromptTemplate(
    input_variables=["action", "subject", "limit"],
    template="Please {action} the concept of {subject} in exactly {limit} words."
)

inputs = [
    {"action": "summarize", "subject": "Blockchain", "limit": "50"},
    {"action": "critique", "subject": "Web3", "limit": "100"},
    {"action": "define", "subject": "NFTs", "limit": "20"},
    {"action": "expand on", "subject": "Smart Contracts", "limit": "200"},
    {"action": "simplify", "subject": "Ethereum", "limit": "30"}
]

for i in inputs:
    print(reusable_tmplt.format(**i))

Please summarize the concept of Blockchain in exactly 50 words.
Please critique the concept of Web3 in exactly 100 words.
Please define the concept of NFTs in exactly 20 words.
Please expand on the concept of Smart Contracts in exactly 200 words.
Please simplify the concept of Ethereum in exactly 30 words.
